## 1. Loading the data
We'll use the same data for sentiement analysis used in previous sessions. So first, we need to mount our Drive account.

In [ ]:
# Mount Drive files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# install libraries
!pip install tensorflow==2.15.0
!pip install tf-keras==2.15.1
!pip install datasets

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

# import load_dataset from dataset
from datasets import load_dataset

import numpy as np
import pandas as pd
import re

import tensorflow as tf
from sklearn.utils import shuffle

## for replicability of results
tf.keras.utils.set_random_seed(2)

# custom data loader path CHANGE home TO YOUR DIRECTORY
home = '/content/drive/MyDrive/ColabNotebooksHapLab/TIA-2024-2025/'
sst_home = home + 'utils/sst_custom_loader.py'
# load data with load_dataset and our custom data loader script
data = load_dataset(sst_home)

def pretty_print(example):
    print('Label: {}\nText: {}'.format(example['label'], example['text']))

training_set = pd.DataFrame(data['train'])
dev_set = pd.DataFrame(data['validation'])
test_set = pd.DataFrame(data['test'])

# Shuffle dataset
training_set = shuffle(training_set)
dev_set = shuffle(dev_set)
test_set = shuffle(test_set)

# Obtain text and label vectors
train_texts = training_set.text
train_labels = training_set.label

dev_texts = dev_set.text
dev_labels = dev_set.label

test_texts = test_set.text
test_labels = test_set.label


print('Training size: {}'.format(len(training_set)))
print('Dev size: {}'.format(len(dev_set)))
print('Test size: {}'.format(len(test_set)))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.2/475.2 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 7.7 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 1.16.0
    Uninstalling wrapt-1.16.0:
      Successfully uninstalled wrapt-1.16.0
  Attempting uninstall: ml-dtypes
    Found existing installation: ml-dtypes 0.4.1
    Uninstalling ml-dtypes-0.4.1:
      Successfully uninstalled ml-dtypes-0.4.1
  Attempting uninstall: keras
    Found existing installation: keras 3.5.0
    Uninstalling keras-3.5.0:
      Successfully uninstalled keras-3.5.0
  Attempting uninstall: tensorboard
    Found existing installation

## 2. Installing and seting up the Transformers library

In [ ]:
# https://blog.tensorflow.org/2019/11/hugging-face-state-of-art-natural.html
!pip install transformers

Once the transformers library is installed, we can use it directly just creating three object of two classes:

- **The tokenizer class**: the tokenizer class takes care of converting input  string in tensors of integers which are indices in a model vocabulary. The tokenization varies according to the model, therefore each model has its own tokenizer.

- **The model class**: the model class holds the neural network modeling logic itself. When using a TensorFlow model, it inherits from tf.keras.layers. Layer which means it can be used very simply by the Keras’ fit API or make more complicated stuff.

There is also **configuration class** that is also required unless you are not using the default values. With the configuration class we indicate everything related to hyperparaters such as number of layers, dropout and so on. Below is an example of a BERT configuration file, for the pre-trained weights bert-base-cased.

```
{
  "attention_probs_dropout_prob": 0.1,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "max_position_embeddings": 512,
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "type_vocab_size": 2,
  "vocab_size": 28996
}
```


In [ ]:
from transformers import TFBertForSequenceClassification, BertTokenizer

model = TFBertForSequenceClassification.from_pretrained("bert-base-cased")
tokenizer = BertTokenizer.from_pretrained("bert-base-cased")


Next we will define two helper function to 1) extract features from the tokenizer (`convert_examples_to_features`) and 2) convert the features to `tf.data.Dataset` object class (`convert_features_to_tf_dataset`). `tf.data.Dataset` is a convinient API that helps managing and iterating in efficient way the input and output data of the model.  For more information you can check the API in tensorflow web page: https://www.tensorflow.org/api_docs/python/tf/data/Dataset.

It might be interesing to read first this glosary to better understand some terminology: https://huggingface.co/transformers/v2.11.0/glossary.html#attention-mask

And afterwards read the documentation related to the tokenizer. batch_encode_plus https://huggingface.co/transformers/v2.11.0/main_classes/tokenizer.html

In [ ]:
from transformers import InputFeatures

def convert_examples_to_features(texts, labels):
  labels = list(labels)
  batch_encoding = tokenizer.batch_encode_plus(texts, max_length=128, padding=True, truncation=True)

  features = []
  for i in range(len(texts)):
      inputs = {k: batch_encoding[k][i] for k in batch_encoding}

      feature = InputFeatures(**inputs, label=labels[i])
      features.append(feature)

  for i, example in enumerate(texts[:5]):
      print("*** Example ***")
      print("text: %s" % (example))
      print("features: %s" % features[i])

  return features

def convert_features_to_tf_dataset(features):
  def gen():
      for ex in features:
          yield (
              {
                  "input_ids": ex.input_ids,
                  "attention_mask": ex.attention_mask,
                  "token_type_ids": ex.token_type_ids,
              },
              ex.label,
          )
  dataset = tf.data.Dataset.from_generator(gen,
                                           ({"input_ids": tf.int32, "attention_mask": tf.int32, "token_type_ids": tf.int32}, tf.int64),
                                           (
                                               {
                                                "input_ids": tf.TensorShape([None]),
                                                "attention_mask": tf.TensorShape([None]),
                                                "token_type_ids": tf.TensorShape([None])
                                                },
                                            tf.TensorShape([]),
                                            ))
  return dataset

Let's preprocess the training and development sets. Note that we use the `tf.data.Dataset` API to set the batch size to 32.

In [ ]:
train_features = convert_examples_to_features(train_texts, train_labels)
train_dataset = convert_features_to_tf_dataset(train_features)

dev_features = convert_examples_to_features(dev_texts, dev_labels)
dev_dataset = convert_features_to_tf_dataset(dev_features)

train_dataset = train_dataset.shuffle(100).batch(32)
dev_dataset = dev_dataset.batch(32)

The results of the tokenizer can be seen in the cell below. There are some differences if vary the tokenizer, but most of them provide the following information.

- `input_ids`: list of token ids to be fed to a model. Remember that the tokens are subwords, and new tokens are included to indicate sentence separation or ending (`[SEP]`) as well as `[CLS]` token that allow the sentence classification .

- `token_type_ids`: list of token type ids to be fed to a model.

- `attention_mask`: list of indices specifying which tokens should be attended to by the model


In [ ]:
# take one bacth of 32 examples.
instance = list(train_dataset.take(1).as_numpy_iterator())
print(instance)

## 3. Understanding the tokenizer

When we preprocess the input text to be fed in BERT like encoder, we tipically make three steps:

1. Break words into tokens (subwords).
2. Add the special tokens such as `[CLS]` and `[SEP]`. These special tokens are already included in the model's vocabulary, so they have their own token id.
3. Substitute the tokens with their corresponding ids. After this step will get the proper shape for BERT.

The code cell bellow shows the results of the three steps.




In [ ]:
sentence1 = "a visually stunning rumination on love."
sentence2 = "There ought to be a directing license, so that Ed Burns can have his revoked."

# Tokenize sentence
sentence1_tokenized = tokenizer.tokenize(sentence1)
print('0. INPUT SENTENCE: {}'.format(sentence1))
print('1. TOKENIZED SENTENCE: {}'.format(sentence1_tokenized))

# Add Special tokens
sentence1_tokenized_with_special_tokens = ['[CLS]'] + sentence1_tokenized + ['[SEP]']
print('2. ADD [CLS], [SEP]: {}'.format(sentence1_tokenized_with_special_tokens))
sentence1_ids = tokenizer.convert_tokens_to_ids(sentence1_tokenized_with_special_tokens)

# Convert to ids
print('3. SENTENCE IDS: {}'.format(sentence1_ids))

### Exercise 1:
- Can you see what happened to "rumination" after the tokenization?
- Can you identify the token ids for [CLS] and [SEP]?

-----

The three steps can be done with `encode` or `batch_encode_plus` functions. The first function takes single string and convert is to ids. Then second function is more convinient to preprocess larger input data. It returns all the requiered information (input_ids, token_type_ids, attention_mask, etc) in a python dictionary.

In [ ]:
# how tokenize and get the token ids with one funtions
sentence1_ids = tokenizer.encode(sentence1, add_special_tokens=True)

print('SENTENCE IDS: {}'.format(sentence1_ids))

# there are more convinient methods to preprocess the input data.
batch_encoding = tokenizer.batch_encode_plus(
        [sentence1], max_length=128, padding=True, truncation=True
    )
print('ENCODE PLUS: {}'.format(batch_encoding))

### Two sentences as input

As you have seen in the theoretical part BERT is a masked language models that learns predicting masked words, and in addition it predicts if next sentence belongs after the first one. That's why BERT's tokenizer is ready to have two sentences as input. This way preprocessing the data is interesting for task like Semantic Textual Similiraty and Natural Language Inference.

In [ ]:
# how tokenize and get the token ids with one funtions
sentence_pair_ids = tokenizer.encode(text=sentence1, text_pair=sentence2, add_special_tokens=True)


# there are more convinient methods to preprocess the input data.
batch_encoding = tokenizer.batch_encode_plus(
        [(sentence1, sentence2)], max_length=128, padding=True, truncation=True,
    )

print("SENTENCE PAIR IDS: {}".format(sentence_pair_ids))


### Exercise 2:
- From the IDs can you say which ids correspond to the first sentence and which to the second one?

## 4. Fine-tune BERT as Sentence Classifier

Here in the next step you will define and train a sentiment analysis model (your dataset has been annotated with sentiment analysis tags and therefore the model will learn that task)

In [ ]:
# Prepare training: Compile tf.keras model with optimizer, loss and learning rate schedule
optimizer = tf.keras.optimizers.Adam(learning_rate=3e-5, epsilon=1e-08, clipnorm=1.0)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metric = tf.keras.metrics.SparseCategoricalAccuracy('accuracy')
model.compile(optimizer=optimizer, loss=loss, metrics=[metric])

# Train and evaluate using tf.keras.Model.fit()
history = model.fit(train_dataset, epochs=3, validation_data=dev_dataset)


In [ ]:
import matplotlib.pyplot as plt

# summarize history for loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'dev'], loc='upper left')
plt.show()

# summarize history for accuracy
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'dev'], loc='upper left')
plt.show()

A nice feature when fine-tuning a large model is that we do not need to train many epochs (the number of epochs depends on the size of the training set). In this case, it looks like just one epoch can be enough.

Once the model is fine-tuned for sentiment analysis we could evaluate it on the test set. In this we need to tokenized and convert to ids the input too.

In [ ]:
test_features = convert_examples_to_features(test_texts, test_labels)
test_dataset = convert_features_to_tf_dataset(test_features)

In [ ]:
test_dataset = test_dataset.batch(32)
instances = list(test_dataset.take(1).as_numpy_iterator())
instances

In [ ]:
model.evaluate(test_dataset)